# Detroit RMS Crime: Bronze-Ingestion

Dieses Notebook unterstützt vollständige Jahresbestände und einen inkrementellen Daily Load über die offizielle ArcGIS-REST-API. Es verwendet die produktive Ingestion aus `detroit_pipeline`, damit Notebook und CLI nicht zwei unterschiedliche Implementierungen besitzen.

**Pipeline-Stufe:** ArcGIS REST → unveränderter Bronze-Snapshot (NDJSON) + Run-Manifest.

## Schritt 1 – Projektpfad und Imports

Die Zelle sucht den Projekt-Root sowohl beim Start im Root als auch direkt aus dem Notebook-Ordner.

In [ ]:
from datetime import date, datetime, timedelta
from pathlib import Path
import json
import sys

# Das Paket liegt seit dem Umzug unter src/crime/. Gesucht wird der Projekt-Root
# sowohl beim Start im Root als auch direkt aus dem Notebook-Ordner.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/detroit_pipeline nicht gefunden')

PACKAGE_ROOT = PROJECT_ROOT / 'src' / 'crime'
sys.path.insert(0, str(PACKAGE_ROOT))

from detroit_pipeline import ArcGISClient
from detroit_pipeline.detroit_arcgis import (
    SOURCE_TIMEZONE,
    build_updated_where,
    build_year_where,
)
from detroit_pipeline.extract_detroit import (
    extract_updated_window,
    extract_year,
    format_load_summary,
)

## Schritt 2 – Lauf konfigurieren

`LOAD_MODE` wählt `FULL` oder `DAILY`. Der Full Load lädt jedes definierte Jahr separat nach `incident_occurred_at` (gemeldeter Tat-/Ereigniszeitpunkt). Der Daily Load selektiert einzelne Offense-Zeilen anhand `updated_at`: neue Reports, neue Delikte unter einer bestehenden `report_number` sowie erneut publizierte oder geänderte Zeilen können dadurch erfasst werden. Ein spätes `updated_at` beweist allein keine fachliche Änderung; die spätere Silver-Stufe muss die fachlichen Feldwerte vergleichen. `RUN_DOWNLOAD` ist absichtlich zunächst `False`.

In [ ]:
LOAD_MODE = "FULL"  # Erlaubt: "FULL" oder "DAILY"
RUN_DOWNLOAD = True  # True = Daten herunterladen, False = nur Zusammenfassung erstellen

# Nur für LOAD_MODE = "FULL"
YEARS = [2024] # Liste der Jahre, für die Daten heruntergeladen werden sollen

# Nur für LOAD_MODE = "DAILY"
DAILY_LOOKBACK_DAYS = 14
DAILY_REFERENCE_DATE = None  # Optional z. B. date(2026, 9, 5) für Backfills

OUTPUT_DIR = PROJECT_ROOT / "data" / "bronze" / "detroit"
BATCH_SIZE = 1000
PAUSE_SECONDS = 0.05

{"load_mode": LOAD_MODE, "run_download": RUN_DOWNLOAD, "output_dir": OUTPUT_DIR}

## Schritt 3 – ArcGIS-Metadaten und erwartete Zeilenzahl prüfen

Vor dem Download werden Schema-Fingerprint, Object-ID-Feld, API-Limit und serverseitiger Count geprüft. Im Daily-Modus ist das Intervall halb-offen und verwendet `updated_at`, damit auch nachträgliche Korrekturen erfasst werden.

In [ ]:
client = ArcGISClient()

layer_info = client.get_layer_info()

load_mode = LOAD_MODE.strip().upper()
if load_mode not in {"FULL", "DAILY"}:
    raise ValueError('LOAD_MODE muss "FULL" oder "DAILY" sein')
if DAILY_LOOKBACK_DAYS <= 0:
    raise ValueError("DAILY_LOOKBACK_DAYS muss größer als 0 sein")

source_checks = []
daily_run_date = DAILY_REFERENCE_DATE or datetime.now(SOURCE_TIMEZONE).date()
daily_window_end = daily_run_date + timedelta(days=1)
daily_window_start = daily_window_end - timedelta(days=DAILY_LOOKBACK_DAYS)

if load_mode == "FULL":
    for year in YEARS:
        where = build_year_where(year)
        source_checks.append({
            "mode": "FULL",
            "year": year,
            "source_count": client.get_count(where),
        })
else:
    where = build_updated_where(daily_window_start, daily_window_end)
    source_checks.append({
        "mode": "DAILY",
        "date_field": "updated_at",
        "window_start": daily_window_start.isoformat(),
        "window_end_exclusive": daily_window_end.isoformat(),
        "source_count": client.get_count(where),
    })

layer_check = {
    "object_id_field": layer_info.object_id_field,
    "max_record_count": layer_info.max_record_count,
    "field_count": len(layer_info.fields),
    "schema_fingerprint": layer_info.schema_fingerprint,
}

{"layer": layer_check, "windows": source_checks}

## Schritt 4 – Vollständigen Bronze-Snapshot laden

Der Extractor fixiert zuerst alle Object-IDs und lädt anschließend genau diese IDs batchweise. Full Loads werden nach `year=...`, Daily Loads nach `run_date=...` partitioniert. Erst nach erfolgreicher Count-Prüfung wird der Bronze-Snapshot veröffentlicht.

In [ ]:
run_results = []

if RUN_DOWNLOAD and load_mode == "FULL":
    for year in YEARS:
        year_output_dir = OUTPUT_DIR / "full" / f"year={year}"
        print(f"\nStarte Bronze-Ingestion für {year} ...")

        bronze_path, manifest_path = extract_year(
            year=year,
            output_dir=year_output_dir,
            batch_size=BATCH_SIZE,
            pause_seconds=PAUSE_SECONDS,
        )

        run_results.append({
            "mode": "FULL",
            "period": str(year),
            "bronze_path": bronze_path,
            "manifest_path": manifest_path,
        })

        print(f"Bronze:  {bronze_path}")
        print(f"Manifest: {manifest_path}")
elif RUN_DOWNLOAD and load_mode == "DAILY":
    daily_output_dir = (
        OUTPUT_DIR / "incremental" / f"run_date={daily_run_date.isoformat()}"
    )
    print(
        f"\nStarte Daily-Ingestion: {daily_window_start} "
        f"bis {daily_window_end} (exklusiv) ..."
    )
    bronze_path, manifest_path = extract_updated_window(
        start_date=daily_window_start,
        end_date_exclusive=daily_window_end,
        output_dir=daily_output_dir,
        batch_size=BATCH_SIZE,
        pause_seconds=PAUSE_SECONDS,
    )
    run_results.append({
        "mode": "DAILY",
        "period": f"{daily_window_start}/{daily_window_end}",
        "bronze_path": bronze_path,
        "manifest_path": manifest_path,
    })
    print(f"Bronze:  {bronze_path}")
    print(f"Manifest: {manifest_path}")
else:
    print("Download deaktiviert. Setze RUN_DOWNLOAD = True.")

## Schritt 5 – Geladene Datensätze je Deliktkategorie prüfen

Die Kategoriesumme muss exakt der Gesamtzahl der geladenen Bronze-Zeilen entsprechen. Fehlende Kategorien werden explizit als `__MISSING__` ausgewiesen.

In [ ]:
import json

for result in run_results:
    manifest = json.loads(
        result["manifest_path"].read_text(encoding="utf-8")
    )

    print(f"\n{'=' * 60}")
    print(f"Modus: {result['mode']}")
    print(f"Zeitraum: {result['period']}")
    print(
        f"Selektion: {manifest['selection_basis']} "
        f"über {manifest['date_field']}"
    )
    print(f"Zeilengranularität: {manifest['record_granularity']}")
    print(f"Source-Löschungen erfasst: {manifest['captures_source_deletions']}")
    print(f"Bronze-Datei: {result['bronze_path']}")
    print(f"{'=' * 60}")

    print(
        format_load_summary(
            manifest["offense_category_counts"],
            manifest["downloaded_count"],
        )
    )

## Ergebnis

Nach erfolgreichem Lauf liegen je Full-Jahr beziehungsweise Daily-Lauf zwei Artefakte vor:

- Full: `full/year=<YEAR>/detroit_rms_crime_<YEAR>_<RUN_ID>.ndjson`
- Daily: `incremental/run_date=<DATE>/detroit_rms_crime_incremental_<START>_<END>_<RUN_ID>.ndjson`
- Zu jeder NDJSON-Datei entsteht ein gleichnamiges `.manifest.json` mit Modus, Wasserzeichen, Counts, Schema-Hash, Checksumme und Laufzeit.

Daily-Fenster überlappen absichtlich. Bronze bewahrt jede geladene Version mit eigener Run-ID. Die spätere Silver-Stufe muss deshalb nach `offense_key` konsolidieren, neue Delikte einer vorhandenen `report_number` separat erhalten und fachliche Änderungen von technischen Neupublikationen unterscheiden. Der Incremental Load erkennt keine gelöschten Source-Records und kann Records mit `updated_at = null` nicht selektieren; dafür bleiben periodische Full-Abgleiche erforderlich.